# Industrial-VLM: Visual Grounding QLoRA Pipeline
**Model**: LLaVA-1.5-7B | **Method**: QLoRA (4-bit NF4) | **Dataset**: MVTec AD (15 categories)

**Upgrade**: Classification → Visual Grounding (Bounding Box Detection)

This notebook runs the complete pipeline with **session-resilient checkpointing**:
1. Data preprocessing (Smart Crop + BBox extraction from ground_truth masks)
2. WandB authentication
3. Zero-shot baseline evaluation (with IoU metrics)
4. QLoRA fine-tuning (r=32, alpha=64, Q-K-V-O + MLP projections)
5. Fine-tuned evaluation (Strict F1: Class + IoU > 0.5)
6. WandB artifact backup (weights + results)
7. Auto-push results to GitHub (fault-tolerant)

**Requirements**: Kaggle GPU T4 x2, MVTec AD dataset attached.

**Session Safety**: Every critical step saves to `/kaggle/working/` (persisted by Kaggle)
and syncs to WandB. If session dies, re-run with `--resume_from_checkpoint`.

## Cell 1: Setup & Environment Initialization

In [ ]:
import os, shutil

REPO_DIR = "/kaggle/working/vlm-industrial-finetuner"

# Clean previous clone if exists to prevent nested directories
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone https://github.com/dvydinh/vlm-industrial-finetuner.git {REPO_DIR}
!pip install -q -r {REPO_DIR}/requirements.txt
!pip install -q wandb -U

print("[CHECKPOINT 1/7] Environment ready.")

## Cell 2: Data Preprocessing (Visual Grounding ETL)
Converts MVTec AD to supervised instruction-tuning JSONL with bounding boxes:
- Smart Crop 336x336 around defect region (with random jitter)
- BBox extraction from `ground_truth/` masks via `cv2.findContours`
- Stratified 80/20 train/test split
- Good samples: full 336x336 resize (no mask available)

**Session Safe**: Processed data saved to `/kaggle/working/processed_data/` (persisted).

In [ ]:
import os

PROCESSED_DIR = "/kaggle/working/processed_data"

# Skip if already processed (session resume scenario)
if os.path.exists(os.path.join(PROCESSED_DIR, "train.jsonl")):
    print("[SKIP] Processed data already exists. Skipping ETL step.")
else:
    !python /kaggle/working/vlm-industrial-finetuner/src/data_builder.py \
        --data_dir /kaggle/input/datasets/ipythonx/mvtec-ad \
        --output_dir {PROCESSED_DIR}

# Verify output
for f in ["train.jsonl", "test.jsonl"]:
    path = os.path.join(PROCESSED_DIR, f)
    if os.path.exists(path):
        lines = sum(1 for _ in open(path))
        print(f"  {f}: {lines} samples")
    else:
        print(f"  ERROR: {f} not found!")

print("[CHECKPOINT 2/7] Data preprocessing complete.")

## Cell 3: Weights & Biases Authentication
Authenticates WandB for remote logging and artifact storage.
Get your API key from https://wandb.ai/authorize

**Session Safe**: WandB continuously syncs training metrics in real-time.

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))
    print("[SYSTEM] WandB logged in using Kaggle Secrets.")
except Exception as e:
    print("[WARN] Could not find Kaggle Secret WANDB_API_KEY. Please replace manually:")
    wandb.login(key="YOUR_WANDB_API_KEY_HERE")

print("[CHECKPOINT 3/7] WandB authenticated.")

## Cell 4: Zero-Shot Baseline Evaluation
Evaluates the base LLaVA-1.5-7B (without LoRA adapters) using the new Visual Grounding metrics.
Reports both basic F1 and strict F1 (Class + IoU > 0.5).

**Session Safe**: Results auto-saved to `/kaggle/working/results/eval_baseline.json`.

In [ ]:
import os

RESULTS_DIR = "/kaggle/working/results"
baseline_json = os.path.join(RESULTS_DIR, "eval_baseline.json")

# Skip if baseline already evaluated (session resume scenario)
if os.path.exists(baseline_json):
    print(f"[SKIP] Baseline results already exist at {baseline_json}")
else:
    !python /kaggle/working/vlm-industrial-finetuner/src/evaluate.py \
        --baseline \
        --test_data /kaggle/working/processed_data

# Quick summary
if os.path.exists(baseline_json):
    import json
    with open(baseline_json) as f:
        r = json.load(f)
    print(f"  Baseline F1 (basic): {r['overall_basic']['f1_macro']:.4f}")
    if 'overall_strict' in r:
        print(f"  Baseline F1 (strict IoU>0.5): {r['overall_strict']['f1_macro']:.4f}")

print("[CHECKPOINT 4/7] Baseline evaluation complete.")

## Cell 5: QLoRA Fine-Tuning (Visual Grounding)
PEFT configuration targeting Q-K-V-O + MLP (gate/up/down) projections.
Expanded LoRA targets increase trainable params from ~38M to ~105M for bbox regression.

**Session Safety Features**:
- Checkpoints saved every 100 steps to `/kaggle/working/lora_weights/`
- WandB syncs checkpoints to cloud (via `WandbArtifactCallback`)
- If session dies, change `RESUME_CHECKPOINT` below to restart from last saved step
- `batch_size=1` to fit expanded LoRA within T4 VRAM

### To Resume After Session Crash:
```python
RESUME_CHECKPOINT = "/kaggle/working/lora_weights/checkpoint-XXX"  # <-- set checkpoint path
```

In [ ]:
import os, glob

OUTPUT_DIR = "/kaggle/working/lora_weights"

# ══════════════════════════════════════════════════════════════
# SESSION RESUME CONFIGURATION
# Set to a checkpoint path to resume, or None for fresh training
# Example: "/kaggle/working/lora_weights/checkpoint-900"
# Or:      "/kaggle/input/notebooks/<user>/<notebook>/lora_weights/checkpoint-900"
RESUME_CHECKPOINT = None
# ══════════════════════════════════════════════════════════════

# Auto-detect: if checkpoints exist in output_dir, offer to resume from latest
if RESUME_CHECKPOINT is None:
    existing = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
                      key=lambda x: int(x.split("-")[-1]))
    if existing:
        RESUME_CHECKPOINT = existing[-1]
        print(f"[AUTO-RESUME] Found existing checkpoint: {RESUME_CHECKPOINT}")

# Build command
cmd = (
    f"python /kaggle/working/vlm-industrial-finetuner/src/train.py "
    f"--dataset /kaggle/working/processed_data "
    f"--output_dir {OUTPUT_DIR} "
    f"--epochs 3 "
    f"--lr 2e-5 "
    f"--batch_size 1 "
    f"--grad_accum 8"
)

if RESUME_CHECKPOINT:
    cmd += f" --resume_from_checkpoint {RESUME_CHECKPOINT}"
    print(f"[SYSTEM] Resuming from: {RESUME_CHECKPOINT}")
else:
    print("[SYSTEM] Starting fresh training run.")

print(f"[CMD] {cmd}")
os.system(cmd)

print("[CHECKPOINT 5/7] Training complete.")

## Cell 6: Fine-Tuned Model Evaluation (Visual Grounding)
Evaluates the combined Base Model + LoRA adapters using strict grounding criteria:
- **F1 Basic**: Binary defect/good classification
- **F1 Strict**: Requires correct defect class AND IoU > 0.5
- **Mean IoU**: Average localization accuracy

**Session Safe**: Results auto-saved to `/kaggle/working/results/eval_finetuned.json`.

In [ ]:
!python /kaggle/working/vlm-industrial-finetuner/src/evaluate.py \
    --model_dir /kaggle/working/lora_weights \
    --test_data /kaggle/working/processed_data

# Quick comparison summary
import json, os

RESULTS_DIR = "/kaggle/working/results"
print("\n" + "=" * 60)
print("  BASELINE vs FINE-TUNED COMPARISON")
print("=" * 60)

for tag in ["baseline", "finetuned"]:
    path = os.path.join(RESULTS_DIR, f"eval_{tag}.json")
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)
        basic = r.get('overall_basic', r.get('overall', {}))
        strict = r.get('overall_strict', {})
        print(f"\n  [{tag.upper()}]")
        print(f"    F1 Basic  : {basic.get('f1_macro', 'N/A')}")
        if strict:
            print(f"    F1 Strict : {strict.get('f1_macro', 'N/A')}")
            print(f"    Mean IoU  : {strict.get('mean_iou', 'N/A')}")

print("\n" + "=" * 60)
print("[CHECKPOINT 6/7] Evaluation complete.")

## Cell 7: WandB Artifact Backup + GitHub Push (Fault-Tolerant)
1. **WandB Artifact**: Uploads weights + results to cloud (FIRST, highest priority)
2. **GitHub Push**: Pushes results to repo (best-effort, won't crash if fails)

**Design**: WandB backup runs BEFORE GitHub push. If GitHub fails, data is still safe in WandB.

In [ ]:
import wandb, subprocess, os

# ═══════════════════════════════════════════════════════
# STEP 1: WandB Artifact Backup (CRITICAL - runs first)
# ═══════════════════════════════════════════════════════
try:
    print("[1/2] Uploading workspace to WandB...")
    run = wandb.init(project="vlm-industrial-finetuner", name="session_archival")
    
    artifact = wandb.Artifact(name="full-kaggle-workspace", type="backup")
    
    lora_dir = "/kaggle/working/lora_weights"
    results_dir = "/kaggle/working/results"
    
    if os.path.exists(lora_dir):
        artifact.add_dir(lora_dir, name="lora_weights")
    if os.path.exists(results_dir):
        artifact.add_dir(results_dir, name="results_and_logs")
    
    run.log_artifact(artifact)
    run.finish()
    print("[SUCCESS] WandB artifact uploaded. Data is SAFE in cloud.")
except Exception as e:
    print(f"[ERROR] WandB backup failed: {e}")

# ═══════════════════════════════════════════════════════
# STEP 2: GitHub Push (Best-effort, won't crash pipeline)
# ═══════════════════════════════════════════════════════
try:
    print("\n[2/2] Pushing results to GitHub...")
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
    
    REPO_DIR = "/kaggle/working/vlm-industrial-finetuner"
    
    subprocess.run(["git", "config", "--global", "user.email", "dvydinh@users.noreply.github.com"], check=True)
    subprocess.run(["git", "config", "--global", "user.name", "dvydinh"], check=True)
    
    os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
    subprocess.run(f"cp /kaggle/working/results/*.json {REPO_DIR}/results/", shell=True)
    subprocess.run(f"cp /kaggle/working/results/*.csv {REPO_DIR}/results/", shell=True)
    
    subprocess.run(["git", "add", "results/"], cwd=REPO_DIR)
    subprocess.run(["git", "commit", "-m", "results: auto-upload visual grounding evaluation metrics"], cwd=REPO_DIR)
    result = subprocess.run(
        ["git", "push", f"https://{GITHUB_TOKEN}@github.com/dvydinh/vlm-industrial-finetuner.git", "main"],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    if result.returncode == 0:
        print("[SUCCESS] Results pushed to GitHub.")
    else:
        print(f"[WARN] Git push failed (non-fatal): {result.stderr[:200]}")
        print("Results are still safe in WandB and in /kaggle/working/results/")
except Exception as e:
    print(f"[WARN] GitHub sync failed (non-fatal): {e}")
    print("Results are still safe in WandB and in /kaggle/working/results/")

print("\n[CHECKPOINT 7/7] Pipeline complete!")
print("All results saved to: /kaggle/working/results/")
print("All artifacts backed up to WandB.")